In [4]:
import os, sys, json, re
from datetime import datetime
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

# ---------------- basic config ----------------
DEFAULT_MODEL_DIR = r"models_HierGenderFamilies\Men"  # hange if you want a different default
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASSUME_DAYS_BETWEEN_MARKS = 7  #used when dates are omitted

# ill be set dynamically from checkpoint
SEASONLABEL_ID = None        #{"Indoors": id, "Outdoors": id, "PAD": id_if_present}
SEASONLABEL_VOCAB = None     #2 (no PAD in ckpt) or 3 (PAD+Indoors+Outdoors in ckpt)

# ---------------- model ----------------
class HierGenderFamilies(nn.Module):
    """
    Season-LSTM -> label-embed concat -> AcrossSeason-LSTM -> per-event head.
    head_variant:
      - "mlp32" : Linear(hid->32) + ReLU + Linear(32->1)
      - "lnlin" : LayerNorm(hid) + Linear(hid->1)
      - "linear": Linear(hid->1)
    """
    def __init__(self, in_dim, hid_season, hid_across, n_events, emb_label_dim, label_vocab, head_variant="mlp32"):
        super().__init__()
        self.season_lstm = nn.LSTM(input_size=in_dim, hidden_size=hid_season, batch_first=True)
        self.label_emb   = nn.Embedding(label_vocab, emb_label_dim)
        self.across_lstm = nn.LSTM(input_size=hid_season + emb_label_dim,
                                   hidden_size=hid_across, batch_first=True)

        def make_head(variant: str):
            if variant == "lnlin":
                return nn.Sequential(nn.LayerNorm(hid_across),
                                     nn.Linear(hid_across, 1))
            elif variant == "linear":
                return nn.Sequential(nn.Linear(hid_across, 1))
            else:  # "mlp32"
                return nn.Sequential(nn.Linear(hid_across, 32),
                                     nn.ReLU(),
                                     nn.Linear(32, 1))
        self.heads = nn.ModuleList([make_head(head_variant) for _ in range(n_events)])

    def forward(self, x, l_season, lab_ids, evt_id):
        """
        x: [B,K,L,F]   (F=3 features: value, days_since_prev, wind)
        l_season: [B,K] (valid lengths per season)
        lab_ids: [B,K]  (label ids per season; PAD rows get 0 when ckpt lacks PAD)
        evt_id: [B]     (which event head to use; defined by last season in window)
        """
        B, K, L, F = x.shape

        # ---- Season (mark-level) encoder ----
        x_flat = x.view(B*K, L, F)
        l_flat = l_season.view(B*K).clamp(min=1)  # pack() disallows zero
        packed = pack_padded_sequence(x_flat, lengths=l_flat.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.season_lstm(packed)   # h_n: [1, B*K, hid_season]
        season_emb = h_n[-1].view(B, K, -1)      # [B,K,hid_season]

        # ---- Add SeasonLabel embedding ----
        lab_emb = self.label_emb(lab_ids)        # [B,K,emb_label]
        season_plus = torch.cat([season_emb, lab_emb], dim=-1)  # [B,K,hid+emb]

        # ---- Across-season encoder ----
        k_len = (l_season > 0).sum(dim=1).clamp(min=1)
        packed_k = pack_padded_sequence(season_plus, lengths=k_len.cpu(), batch_first=True, enforce_sorted=False)
        _, (h2, _) = self.across_lstm(packed_k)  # h2: [1,B,hid_across]
        fam_emb = h2[-1]                          # [B,hid_across]

        # ---- Event-specific heads ----
        preds = torch.empty(B, device=x.device, dtype=torch.float32)
        for eid in torch.unique(evt_id):
            m = (evt_id == eid)
            preds[m] = self.heads[int(eid)](fam_emb[m]).squeeze(-1)
        return preds

# ---------------- checkpoint loader ----------------
def _detect_head_variant(state: dict) -> str:
    ks = state.keys()
    #3-module MLP heads: keys like heads.0.2.weight/bias exist
    if any(k.startswith("heads.0.2.") for k in ks):
        return "mlp32"
    #Try to guess LayerNorm+Linear by looking at the first submodule params
    w0 = state.get("heads.0.0.weight", None)
    if w0 is not None and getattr(w0, "ndim", 0) == 1:
        return "lnlin"   # LayerNorm stores 1-D gamma/beta first
    return "linear"

def _meta_get(meta: dict, *names, default=None):
    for n in names:
        if n in meta:
            return meta[n]
    return default

def load_model_from_dir(model_dir: str, device: str = DEVICE):
    ckpt_path = os.path.join(model_dir, "model.pth")
    meta_path = os.path.join(model_dir, "meta.json")
    if not (os.path.exists(ckpt_path) and os.path.exists(meta_path)):
        raise FileNotFoundError(f"Expected model.pth and meta.json in {model_dir}")

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    #Load state on CPU first (avoids rare CUDA asserts at load)
    state = torch.load(ckpt_path, map_location="cpu")

    in_dim     = int(_meta_get(meta, "in_dim"))
    hid_season = int(_meta_get(meta, "hid_season"))
    hid_across = int(_meta_get(meta, "hid_across"))
    emb_label  = int(_meta_get(meta, "emb_label"))
    evt_list   = list(_meta_get(meta, "events"))

    #Detect label vocab from embedding weight rows
    lbl_w = state.get("label_emb.weight", None)
    if lbl_w is None:
        raise RuntimeError("Checkpoint missing label_emb.weight")
    n_labels_ckpt = int(lbl_w.shape[0])

    global SEASONLABEL_ID, SEASONLABEL_VOCAB
    if n_labels_ckpt == 2:
        SEASONLABEL_VOCAB = 2
        SEASONLABEL_ID = {"Indoors": 0, "Outdoors": 1, "PAD": 0}
    else:
        SEASONLABEL_VOCAB = n_labels_ckpt
        SEASONLABEL_ID = {"PAD": 0, "Indoors": 1, "Outdoors": 2}

    head_variant = _detect_head_variant(state)
    model = HierGenderFamilies(in_dim, hid_season, hid_across, len(evt_list),
                               emb_label, label_vocab=SEASONLABEL_VOCAB,
                               head_variant=head_variant)
    #Try strict first; fall back to non-strict with a warning
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as e:
        print("⚠️ Strict load failed; retrying non-strict:\n", e, "\n")
        model.load_state_dict(state, strict=False)

    model.to(device)
    model.eval()

    mu_by_event = {str(k): float(v) for k, v in _meta_get(meta, "mu_by_event").items()}
    sd_by_event = {str(k): float(v) for k, v in _meta_get(meta, "sd_by_event").items()}

    seqlen    = int(_meta_get(meta, "seqlen_marks", "seqlen", default=32))
    k_seasons = int(_meta_get(meta, "k_seasons", default=4))

    return model, meta, evt_list, mu_by_event, sd_by_event, seqlen, k_seasons

# ---------------- parsing & formatting helpers ----------------
_TIME_RE = re.compile(r"^\s*(?:(\d+):)?(\d+)(?:\.(\d+))?\s*$")

def parse_time_like(s: str) -> Optional[float]:
    s = str(s).strip()
    m = _TIME_RE.match(s)
    if not m:
        return None
    mm, ss, frac = m.group(1), m.group(2), m.group(3)
    total = 0.0
    if mm is not None:
        total += 60.0 * float(mm)
    total += float(ss)
    if frac:
        total += float("0."+frac)
    return total

def parse_float(s: str) -> Optional[float]:
    try:
        return float(str(s).strip().lower().replace("m/s","").replace("m",""))
    except Exception:
        return None

def parse_date(s: str) -> Optional[datetime]:
    s = (s or "").strip()
    if not s:
        return None
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%b %d, %Y", "%b %d %Y"):
        try:
            return datetime.strptime(s, fmt)
        except Exception:
            pass
    return None

def is_time_event(name: str) -> bool:
    name_l = name.lower()
    time_keywords = ["meters", "hurdles", "relay", "mile", "miles", "steeple", "1500", "3000", "5000", "10000", "10k", "marathon"]
    field_keywords = ["jump", "vault", "shot", "discus", "hammer", "javelin", "weight"]
    if any(k in name_l for k in field_keywords):
        return False
    if any(k in name_l for k in time_keywords):
        return True
    #default guess: if it contains a number + "m" it's likely a track event
    return bool(re.search(r"\d+\s*m", name_l))

def fmt_time(seconds: float) -> str:
    if seconds is None or not np.isfinite(seconds): return "n/a"
    if seconds >= 60:
        m = int(seconds // 60)
        s = seconds - 60*m
        return f"{m}:{s:05.2f}"
    return f"{seconds:.2f}s"

def fmt_dist(meters: float) -> str:
    if meters is None or not np.isfinite(meters): return "n/a"
    return f"{meters:.2f} m"

# ---------------- build one-season window ----------------
def build_single_season_block(
    marks: List[Tuple[Optional[datetime], float, float]],
    L: int,
    indoor_zero_wind: bool,
) -> Tuple[np.ndarray, int]:
    """
    marks: list of (date_or_None, value_native, wind_mps)
    returns X[L,3], valid_len
    """
    if not marks:
        return np.zeros((L,3), dtype="float32"), 0

    if any(d is not None for d,_,_ in marks):
        marks = sorted(marks, key=lambda t: t[0] or datetime(2000,1,1))

    vals, days, winds = [], [], []
    prev_date = None
    for d, v, w in marks:
        vals.append(float(v))
        if d is None:
            days.append(float(0 if prev_date is None else ASSUME_DAYS_BETWEEN_MARKS))
        else:
            if prev_date is None: days.append(0.0)
            else: days.append(float(max(0, (d - prev_date).days)))
            prev_date = d
        winds.append(0.0 if indoor_zero_wind else float(w or 0.0))

    X = np.stack([np.array(vals, dtype="float32"),
                  np.array(days, dtype="float32"),
                  np.array(winds, dtype="float32")], axis=1)  # [n,3]
    n = len(vals)
    if n > L:
        X = X[-L:]
        n = L
    else:
        pad = np.zeros((L-n, 3), dtype="float32")
        X = np.vstack([pad, X])
    return X, n

def build_inputs_for_window(
    season_label: str, event_name: str,
    marks: List[Tuple[Optional[datetime], float, float]],
    seqlen: int, k_seasons: int, evt_list: List[str], device: str
):
    #final-season block (the season you're in right now)
    X_last, n_last = build_single_season_block(
        marks, seqlen, indoor_zero_wind=(season_label == "Indoors")
    )

    #pad K-1 seasons on the left (empty history)
    blocks = [np.zeros((seqlen,3), dtype="float32") for _ in range(k_seasons-1)] + [X_last]
    lens   = [0]*(k_seasons-1) + [n_last]

    #label mapping depends on checkpoint vocab
    if SEASONLABEL_VOCAB == 2:
        lab_map = {"Indoors": 0, "Outdoors": 1}
        labs = [0]*(k_seasons-1) + [lab_map[season_label]]   #PAD shares 0
    else:
        lab_map = {"PAD": 0, "Indoors": 1, "Outdoors": 2}
        labs = [lab_map["PAD"]] * (k_seasons-1) + [lab_map[season_label]]

    if event_name not in evt_list:
        raise ValueError(f"Event '{event_name}' not in trained events. Pick from: {', '.join(evt_list[:30])} ...")

    X   = np.stack(blocks, axis=0)[None, ...]            # [1,K,L,3]
    Lk  = np.array(lens, dtype=np.int64)[None, ...]      # [1,K]
    LAB = np.array(labs, dtype=np.int64)[None, ...]      # [1,K]
    E   = np.array([evt_list.index(event_name)], dtype=np.int64)

    X_t  = torch.from_numpy(X).to(device)
    Lk_t = torch.from_numpy(Lk).to(device)
    LAB_t= torch.from_numpy(LAB).to(device)
    E_t  = torch.from_numpy(E).to(device)
    return X_t, Lk_t, LAB_t, E_t

# ---------------- prompts ----------------
def prompt_model_dir(default=DEFAULT_MODEL_DIR) -> str:
    p = input(f"Path to model folder [{default}]: ").strip()
    return p or default

def prompt_season_label() -> str:
    while True:
        s = input("Predict NEXT season from (Indoors/Outdoors): ").strip().title()
        if s in ("Indoors","Outdoors"):
            return s
        print("  Please type exactly Indoors or Outdoors.")

def prompt_event(evt_list: List[str]) -> str:
    print("\nAvailable events (first 40 shown):")
    for i, e in enumerate(evt_list[:40], 1):
        print(f"  {i:2d}. {e}")
    raw = input("Type exact event name or its number: ").strip()
    if raw.isdigit():
        idx = int(raw) - 1
        if 0 <= idx < len(evt_list):
            return evt_list[idx]
    if raw in evt_list:
        return raw
    #fuzzy contains
    cand = [e for e in evt_list if raw.lower() in e.lower()]
    if len(cand) == 1:
        print(f"Using '{cand[0]}'")
        return cand[0]
    if cand:
        print("Ambiguous. Matches:", ", ".join(cand[:10]))
    raise ValueError("Event not recognized.")

def prompt_marks_structured(season_label: str, event_name: str) -> List[Tuple[Optional[datetime], float, float]]:
    print("\nEnter your marks for THIS season. I'll guide you field-by-field.")
    print("Dates are optional; indoor wind is forced to 0.0 automatically.")
    time_mode = is_time_event(event_name)
    if time_mode:
        print("Detected a TIME event. Value examples: 10.55  or  1:50.32")
    else:
        print("Detected a FIELD event. Value examples: 13.72  or  13.72m")

    while True:
        n_s = input("How many marks will you enter (e.g., 4)? ").strip()
        if n_s.isdigit() and int(n_s) > 0:
            N = int(n_s)
            break
        print("  Please enter a positive integer.")
    marks = []
    for i in range(1, N+1):
        print(f"\nMark {i}/{N}:")
        ds = input("  Date (YYYY-MM-DD) [optional, Enter=skip]: ").strip()
        d = parse_date(ds) if ds else None

        vs = input("  Value (time or distance): ").strip()
        v = parse_time_like(vs) if time_mode else None
        if v is None:
            v = parse_float(vs)
        if v is None:
            print("  Could not parse value; try again.")
            return prompt_marks_structured(season_label, event_name)

        if season_label == "Indoors":
            w = 0.0
            print("  Wind set to 0.0 for indoors.")
        else:
            ws = input("  Wind m/s (e.g., +1.2) [optional, Enter=0]: ").strip().replace("m/s","").replace("(","").replace(")","")
            w = parse_float(ws) if ws else 0.0
            if w is None:
                w = 0.0
        marks.append((d, float(v), float(w)))
    return marks

# ---------------- main ----------------
def main():
    #Safer defaults for deterministic CPU inference if needed
    torch.set_grad_enabled(False)

    model_dir = prompt_model_dir()
    model, meta, evt_list, mu_by_event, sd_by_event, seqlen, k_seasons = load_model_from_dir(model_dir, DEVICE)

    gender = meta.get("gender", "Unknown")
    print(f"\nLoaded model for gender: {gender}")
    print(f"Events in this model: {len(evt_list)}")
    print(f"SeqLen={seqlen}, K={k_seasons}, Device={DEVICE}")

    season_label = prompt_season_label()
    event_name   = prompt_event(evt_list)
    marks        = prompt_marks_structured(season_label, event_name)

    #Build inputs (K-1 empty seasons + your current season block)
    X, Lk, LAB, E = build_inputs_for_window(season_label, event_name, marks,
                                            seqlen, k_seasons, evt_list, DEVICE)

    #Predict z, then de-normalize
    with torch.no_grad():
        yhat_z = model(X, Lk, LAB, E).cpu().numpy().reshape(-1)[0]

    #Map back to native units
    mu = mu_by_event[event_name]
    sd = sd_by_event[event_name]
    yhat = yhat_z * sd + mu

    #Pretty print with guessed units....
    print("\n================= RESULT =================")
    print(f"Event: {event_name}   From season: {season_label}")
    print(f"Marks entered: {len(marks)}  (seqlen={seqlen}, K={k_seasons})")
    if is_time_event(event_name):
        print(f"Predicted NEXT-season peak: {fmt_time(yhat)}  (z={yhat_z:+.3f})")
    else:
        print(f"Predicted NEXT-season peak: {fmt_dist(yhat)}  (z={yhat_z:+.3f})")
    print("==========================================\n")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\nCanceled by user.")



Loaded model for gender: Men
Events in this model: 28
SeqLen=32, K=4, Device=cuda

Available events (first 40 shown):
   1. 10,000 Meters
   2. 100 Meters
   3. 1000 Meters
   4. 110 Hurdles
   5. 1500 Meters
   6. 200 Meters
   7. 300 Meters
   8. 3000 Meters
   9. 3000 Steeplechase
  10. 400 Hurdles
  11. 400 Meters
  12. 500 Meters
  13. 5000 Meters
  14. 55 Meters
  15. 60 Hurdles
  16. 60 Meters
  17. 600 Meters
  18. 800 Meters
  19. Discus
  20. Hammer
  21. High Jump
  22. Javelin
  23. Long Jump
  24. Mile
  25. Pole Vault
  26. Shot Put
  27. Triple Jump
  28. Weight Throw

Enter your marks for THIS season. I'll guide you field-by-field.
Dates are optional; indoor wind is forced to 0.0 automatically.
Detected a FIELD event. Value examples: 13.72  or  13.72m

Mark 1/1:

================= RESULT =================
Event: Triple Jump   From season: Outdoors
Marks entered: 1  (seqlen=32, K=4)
Predicted NEXT-season peak: 11.49 m  (z=-1.794)

